# Clase 8 · Series Temporales II — Live Coding
## Torneo de Forecasting — `ventas_mensuales.csv`

Diplomado de Python y Análisis de Datos · Universidad Marista

**Cómo usar este notebook:**
Cada celda de código está **comentada línea por línea**. Durante la clase,
ve descomentando (quitando el `# ` al inicio de cada línea) según avancemos
en la presentación. Las celdas de texto (Markdown) indican en qué paso de
la presentación estamos.

Dataset: 60 meses de historia de ventas mensuales (`fecha`, `ventas`,
`promociones`, `unidades`).


## Paso 0 — Librerías
Descomentar al iniciar la sesión.

In [ ]:
# Paso 0: Librerías necesarias para todo el notebook
#
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
#
# from sklearn.metrics import mean_absolute_error, mean_squared_error
#
# from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
# from statsmodels.tsa.arima.model import ARIMA
#
# pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Paso 1 — Cargar los datos
(Correspondiente a *Paso 1: cargar* de la presentación)

In [ ]:
# Paso 1: cargar el csv y preparar el índice temporal
#
# df = pd.read_csv("../data/ventas_mensuales.csv")
#
# df["fecha"] = pd.to_datetime(df["fecha"])
#
# df = df.sort_values("fecha")
#
# df = df.set_index("fecha")
#
# print(df.shape)
# df.head()

## Paso 2 — Visualizar la serie
(*Paso 2: visualizar*). Antes de modelar, debemos comprender la serie.

In [ ]:
# Paso 2: graficar la serie completa
#
# plt.figure(figsize=(12, 5))
#
# plt.plot(df.index, df["ventas"])
#
# plt.title("Ventas mensuales")
# plt.xlabel("Fecha")
# plt.ylabel("Ventas")
#
# plt.show()

### Pregunta para el grupo
- ¿La serie muestra tendencia?
- ¿Hay estacionalidad visible (patrón que se repite cada 12 meses)?
- ¿Hay algún valor atípico?

## Paso 3 — Train / Test temporal
(*Paso 3: Train/Test*). Reservamos los últimos 12 meses como si fueran el futuro.

In [ ]:
# Paso 3: dividir train/test respetando el orden temporal
#
# train = df.iloc[:-12].copy()
#
# test = df.iloc[-12:].copy()
#
# print("Train:", train.shape)
# print("Test:", test.shape)
#
# print(train.index.min(), train.index.max())
# print(test.index.min(), test.index.max())

## Paso 4 — Baseline: Naive Forecast
(*Paso 4: Naive*). $\hat{Y}_{t+1} = Y_t$

In [ ]:
# Paso 4: Naive Forecast — repetir el último valor observado
#
# ultimo = train["ventas"].iloc[-1]
#
# pred_naive = [ultimo] * len(test)
#
# print(pred_naive[:5])

## Paso 5 — Seasonal Naive
(*Paso 5: Seasonal Naive*). $\hat{Y}_t = Y_{t-12}$

In [ ]:
# Paso 5: Seasonal Naive — usar el mismo mes del año anterior
#
# pred_seasonal = train["ventas"].iloc[-12:].values
#
# print(pred_seasonal)

### Extra — Media móvil (opcional, referencia adicional)

In [ ]:
# Extra: pronóstico con el promedio de los últimos 3 meses
#
# promedio_reciente = train["ventas"].tail(3).mean()
#
# pred_ma = [promedio_reciente] * len(test)
#
# print(promedio_reciente)

## Paso 6 — Simple Exponential Smoothing (SES)
(*Paso 6: SES*).

In [ ]:
# Paso 6: Suavizamiento exponencial simple
#
# modelo_ses = SimpleExpSmoothing(train["ventas"])
#
# ajuste_ses = modelo_ses.fit()
#
# pred_ses = ajuste_ses.forecast(len(test))
#
# print(pred_ses)

## Paso 7 — Holt (nivel + tendencia)
(*Paso 7: Holt*).

In [ ]:
# Paso 7: Holt — nivel + tendencia
#
# modelo_holt = Holt(train["ventas"])
#
# ajuste_holt = modelo_holt.fit()
#
# pred_holt = ajuste_holt.forecast(len(test))
#
# print(pred_holt)

## Paso 8 — Holt-Winters (nivel + tendencia + estacionalidad)
(*Paso 8: Holt-Winters*).

In [ ]:
# Paso 8: Holt-Winters aditivo, estacionalidad de 12 meses
#
# modelo_hw = ExponentialSmoothing(
#     train["ventas"],
#     trend="add",
#     seasonal="add",
#     seasonal_periods=12
# )
#
# ajuste_hw = modelo_hw.fit()
#
# pred_hw = ajuste_hw.forecast(len(test))
#
# print(pred_hw)

## Paso 9 — ARIMA básico
(*Paso 9: ARIMA*). Introducción conceptual, order=(1,1,1).

In [ ]:
# Paso 9: ARIMA(1,1,1) como acercamiento introductorio
#
# modelo_arima = ARIMA(train["ventas"], order=(1, 1, 1))
#
# ajuste_arima = modelo_arima.fit()
#
# pred_arima = ajuste_arima.forecast(steps=len(test))
#
# print(pred_arima)

## Comparar modelos
Función de evaluación con MAE y RMSE.

In [ ]:
# Evaluar Naive como ejemplo
#
# def evaluar(real, pred):
#     mae = mean_absolute_error(real, pred)
#     rmse = mean_squared_error(real, pred) ** 0.5
#     return mae, rmse
#
# mae_naive, rmse_naive = evaluar(test["ventas"], pred_naive)
# print(mae_naive, rmse_naive)

In [ ]:
# Tabla comparativa del torneo (MAE, RMSE, MAPE)
#
# def mape(real, pred):
#     real = np.array(real)
#     pred = np.array(pred)
#     return np.mean(np.abs((real - pred) / real)) * 100
#
# resultados = pd.DataFrame({
#     "Modelo": [
#         "Naive",
#         "Seasonal Naive",
#         "Media movil",
#         "SES",
#         "Holt",
#         "Holt-Winters",
#         "ARIMA"
#     ],
#     "MAE": [
#         evaluar(test["ventas"], pred_naive)[0],
#         evaluar(test["ventas"], pred_seasonal)[0],
#         evaluar(test["ventas"], pred_ma)[0],
#         evaluar(test["ventas"], pred_ses)[0],
#         evaluar(test["ventas"], pred_holt)[0],
#         evaluar(test["ventas"], pred_hw)[0],
#         evaluar(test["ventas"], pred_arima)[0],
#     ],
#     "RMSE": [
#         evaluar(test["ventas"], pred_naive)[1],
#         evaluar(test["ventas"], pred_seasonal)[1],
#         evaluar(test["ventas"], pred_ma)[1],
#         evaluar(test["ventas"], pred_ses)[1],
#         evaluar(test["ventas"], pred_holt)[1],
#         evaluar(test["ventas"], pred_hw)[1],
#         evaluar(test["ventas"], pred_arima)[1],
#     ],
#     "MAPE": [
#         mape(test["ventas"], pred_naive),
#         mape(test["ventas"], pred_seasonal),
#         mape(test["ventas"], pred_ma),
#         mape(test["ventas"], pred_ses),
#         mape(test["ventas"], pred_holt),
#         mape(test["ventas"], pred_hw),
#         mape(test["ventas"], pred_arima),
#     ],
# })
#
# resultados = resultados.sort_values("MAE")
#
# resultados

## Visualizar: Real vs Pronóstico
Usando el mejor modelo (ajustar la variable `pred_ganador` según el torneo).

In [ ]:
# Graficar train, real y pronóstico del modelo ganador
#
# pred_ganador = pred_hw  # cambiar por el modelo con menor MAE
#
# plt.figure(figsize=(12, 5))
#
# plt.plot(train.index, train["ventas"], label="Train")
# plt.plot(test.index, test["ventas"], label="Real")
# plt.plot(test.index, pred_ganador, label="Forecast")
#
# plt.legend()
# plt.title("Real vs Pronostico")
# plt.show()

## Analizar residuos
¿El modelo está sesgado? ¿Hay estructura remanente en los errores?

In [ ]:
# Calcular y graficar los residuos del modelo ganador
#
# residuos = test["ventas"] - pred_ganador
#
# plt.figure(figsize=(12, 4))
# plt.plot(residuos)
# plt.axhline(0, color="black")
# plt.title("Errores del pronostico")
# plt.show()
#
# print("Error promedio (sesgo):", residuos.mean())

## Cierre
- ¿Qué modelo ganó el torneo (menor MAE / RMSE)?
- ¿El baseline (Naive / Seasonal Naive) fue competitivo?
- ¿Los residuos muestran sesgo o estructura?
- Puente a la Clase 9: Introducción a Machine Learning.